In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.environ.get("HUGGINGFACEHUB_API_TOKEN")



Установка и базовый вызов модели

    Установить LangChain через pip.

    Создать простой скрипт, который отправляет запрос в LLM (например, OpenAI GPT) и выводит ответ.



In [76]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

question = "Who won the FIFA World Cup in the year 1994?"

template = """Question: {question}

Answer: Let's think step by step."""

prompt = PromptTemplate.from_template(template)

repo_id = "gpt2"  # возьмем gpt2 для text-generation

hf_llm = HuggingFaceEndpoint(
    repo_id="gpt2",
    task="text-generation",
    provider="auto",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    temperature=0.8,
    timeout=300  # увеличиваем таймаут, если нужно
)


llm_chain = LLMChain(prompt=prompt, llm=hf_llm)

print(llm_chain.run({"question": question}))


StopIteration: 

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

messages = [HumanMessage(content="Объясни понятие нейронной сети простыми словами.")]
response = llm(messages)
print(response.content)

Нейронная сеть — это компьютерная система, которая пытается имитировать работу человеческого мозга. Она состоит из множества связанных между собой "нейронов" (маленьких элементов), которые обрабатывают информацию.

Представь, что нейронная сеть — это как сеть из компьютеров, которые общаются друг с другом. Когда ты даешь ей какую-то задачу (например, распознать картинку или перевести текст), информация проходит через эту сеть, и каждый нейрон вносит свой вклад в решение задачи.

Нейронные сети обучаются на примерах: их показывают много данных (например, картинки с кошками и собаками), и они постепенно учатся различать эти объекты, находя общие черты. Чем больше данных они обрабатывают, тем лучше становятся в своей работе. В итоге нейронная сеть может принимать решения или делать предсказания на основе нового, ранее не виденного материала.




Простой текстовой prompt

    Написать программу, отправляющую фиксированный prompt (“Напиши небольшое поздравление с днем рождения”) и вывести ответ.



In [78]:
from langchain.chat_models import ChatOpenAI
from langchain import LLMChain
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

prompt = PromptTemplate(
    input_variables=[],
    template="Напиши небольшое поздравление с днем рождения"
)

chain = LLMChain(llm=llm, prompt=prompt)

response = chain.run({})
print(response)

Дорогой(ая) [Имя]!

С днем рождения! Желаю тебе здоровья, счастья и исполнения всех самых заветных желаний. Пусть каждый новый день приносит радость и вдохновение, а рядом будут только верные друзья и близкие. Наслаждайся каждым моментом и смело иди к своим целям!

С наилучшими пожеланиями,  
[Твое имя]




Использование PromptTemplate

    Создать шаблон prompt с параметром (например, {имя}).

    Написать скрипт, который подставляет в шаблон разные имена и генерирует поздравления.



In [79]:
from langchain import LLMChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

prompt = PromptTemplate(
    input_variables=["name"],
    template="Напиши небольшое поздравление с днем рождения для {name}"
)

chain = LLMChain(llm=llm, prompt=prompt)

response = chain.run({"name": "Dima"})
print(response)

Дорогой Дима! 

Поздравляю тебя с днем рождения! Желаю, чтобы каждый день приносил радость и новые возможности, чтобы мечты сбывались, а рядом были верные друзья. Пусть жизнь дарит яркие моменты и вдохновение! 

Счастья, здоровья и удачи тебе во всем! 

С наилучшими пожеланиями,  
[Твое имя]




Создание цепочки (Chain) из двух шагов

    Сделать цепочку: сначала сгенерировать тему письма, затем — письмо на заданную тему.

    Вывести результат обоих шагов.



In [88]:
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

subject_prompt = PromptTemplate(
    input_variables=["имя"],
    template="Придумай короткую тему письма с поздравлением с днем рождения для {имя}"
)
subject_chain = LLMChain(llm=llm, prompt=subject_prompt, output_key="тема")

body_prompt = PromptTemplate(
    input_variables=["тема", "имя"],
    template="Напиши письмо с поздравлением на тему: '{тема}' для {имя}"
)
body_chain = LLMChain(llm=llm, prompt=body_prompt, output_key="body")

chain = SequentialChain(
    chains=[subject_chain, body_chain],
    input_variables=["имя"],
    output_variables=["тема", "body"],
    verbose=True)

response = chain({"имя": "Vova"})
# print(response)
print("Тема:", response["тема"])
print("Письмо:", response["body"])



> Entering new SequentialChain chain...

> Finished chain.
Тема: "С Днем Рождения, Вова! Пусть каждый миг будет ярким!"
Письмо: Дорогой Вова!

С Днем Рождения! 🎉

Сегодня твой особенный день, и я хочу пожелать тебе, чтобы каждый миг твоей жизни был ярким и запоминающимся! Пусть каждый новый день приносит море положительных эмоций, захватывающих событий и вдохновения для новых свершений.

Ты — удивительный человек, и я уверен(а), что впереди у тебя много интересных возможностей и свершений. Не бойся мечтать и следовать своим желаниям, ведь жизнь полна сюрпризов и неожиданностей!

Пусть рядом будут верные друзья, поддержка близких и множество приятных моментов. Желаю здоровья, счастья и успехов во всех начинаниях!

С праздником тебя, Вова! Пусть этот день будет наполнен радостью и весельем!

С наилучшими пожеланиями,  
[Твое имя]




Обработка структурированного вывода

    Заставить модель выдавать ответ в формате JSON.

    Написать парсер, который преобразует строку ответа в Python-словарь.



In [93]:
import json
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

prompt = PromptTemplate(
    input_variables=["имя"],
    template=(
        "Напиши короткое поздравление с днем рождения для {имя} в формате JSON, "
        "где ключи: \"тема\" - короткая тема письма, \"текст\" - само поздравление."
        "верни строгое JSON-значение без пояснений, комментариев и прочего."
        # "Например: {\"тема\": \'...\', \"текст\": \'...\'}"
    )
)

chain = LLMChain(llm=llm, prompt=prompt)

response = chain.run({"имя": "Oleg"})

print("Ответ: ", response)
try:
    parsed = json.loads(response)
    print("Распарсенный ответ: ", parsed)
except json.JSONDecodeError as e:
    print("Ошибка парсинга JSON", e)

Ответ:  {
  "тема": "С днем рождения, Oleg!",
  "текст": "Дорогой Oleg! Поздравляю тебя с днем рождения! Желаю счастья, здоровья и исполнения всех мечт. Пусть каждый день приносит радость и новые возможности!"
}
Распарсенный ответ:  {'тема': 'С днем рождения, Oleg!', 'текст': 'Дорогой Oleg! Поздравляю тебя с днем рождения! Желаю счастья, здоровья и исполнения всех мечт. Пусть каждый день приносит радость и новые возможности!'}




Использование ConversationBufferMemory

    Создать чатбота, который хранит в памяти последние 3 сообщения пользователя.

    Организовать ввод-вывод в цикле, демонстрируя сохранение контекста.



In [95]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="gpt-4o-mini"
)

memory = ConversationBufferMemory(
    max_token_limit=1000,
    return_messages=True,
    memory_key="history"
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print("Чат-бот запущен. Для выхода введите 'exit'.")

while True:
    user_input = input("пользователь: ")

    if user_input.lower() == "exit":
        print("Чат завершен.")
        break

    response = conversation.predict(input=user_input)

    print("Бот: ", response)

    print("\n[Текущий контекст диалога]: ")
    for msg in memory.buffer[-6:]:
        print(f"{msg.type}: {msg.content}")
    print("-" * 40)

Чат-бот запущен. Для выхода введите 'exit'.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Привет, меня зовут Лев, а тебя?
AI:

> Finished chain.
Бот:  Привет, Лев! Я — твой дружелюбный искусственный интеллект. У меня нет имени, но ты можешь звать меня как угодно. Как прошел твой день?

[Текущий контекст диалога]: 
human: Привет, меня зовут Лев, а тебя?
ai: Привет, Лев! Я — твой дружелюбный искусственный интеллект. У меня нет имени, но ты можешь звать меня как угодно. Как прошел твой день?
----------------------------------------


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of 



Локальная работа с векторным поиском (Embedding + Chroma)

    Взять простой набор текстов (например, выжимки из статей).

    Сформировать у них эмбеддинги.

    Реализовать простой поиск похожих текстов по запросу.



In [106]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document

texts = [
    "Искусственный интеллект — область компьютерных наук, изучающая создание умных машин.",
    "Машинное обучение позволяет компьютерам учиться на данных и улучшать свои результаты.",
    "Нейронные сети имитируют работу человеческого мозга для решения различных задач.",
    "Обработка естественного языка помогает компьютерам понимать и генерировать текст.",
    "Облачные вычисления предоставляют доступ к вычислительным ресурсам через интернет."
]

documents = [Document(page_content=t, metadata={"source": f"doc{i}"}) for i, t in enumerate(texts)]

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

chroma_store = Chroma.from_documents(documents, embedding_model, collection_name="example_collection")

query = "Что такое нейронные сети?"
results = chroma_store.similarity_search(query, k=3)

print(f'Поиск по запросу: "{query}"\nРезультаты:')
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}\nИсточник: {doc.metadata['source']}\n")


NameError: name 'LRScheduler' is not defined